# LangChain 03 · 记忆与流式（短期记忆 / 长期记忆 / stream）

这一课把「记忆」和「流式」两件事放在一起讲：**记忆**解决「智能体怎么记得住」，
**流式**解决「智能体怎么把过程一点点吐出来」。四个概念贯穿全课：

| 概念 | 是什么 | 代码形态 |
|---|---|---|
| 短期记忆 checkpointer | 按 `thread_id` 记住本轮会话的历史消息 | `checkpointer=MemorySaver()/PostgresSaver()` |
| 长期记忆 store | 按 `namespace` 存「事实 / 偏好」，跨会话生效 | `store=InMemoryStore()/PostgresStore()` |
| 流式 stream | 把执行过程边跑边吐出来 | `stream_mode="messages"/"updates"/"values"` |
| 事件流 v3 | 官方新一代流式，把每种数据做成「独立投影」 | `stream_events(..., version="v3")` |

> **本 notebook 由 `Agent/02_langchain/` 下 7 个脚本合并而成**：
> `05_短期记忆.py`（原版）、`05_短期记忆_jxsd.py`（完整版）、
> `06_长期记忆.py`（原版）、`06_长期记忆_jxsd.py`（完整版）、
> `07_流式输出.py`（原版）、`07_流式输出_jxsd.py`（完整版）、
> `19_事件流v3_官方补充.py`（官方补充）。

**官方文档**
- 短期记忆：<https://docs.langchain.com/oss/python/langchain/short-term-memory>
- 长期记忆：<https://docs.langchain.com/oss/python/langchain/long-term-memory>
- 流式输出：<https://docs.langchain.com/oss/python/langchain/streaming>
- 事件流 v3：<https://docs.langchain.com/oss/python/langchain/event-streaming>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型 |
| 依赖 | `langchain` / `langgraph` / `langchain-core`（venv 已装） |
| 密钥 | `settings.api_key`（已配置） |
| 前置服务 | 短期/长期记忆**完整版**需要 PostgreSQL（`settings.pg_uri`）；连不上会自动打印 `[跳过]` 降级，其余小节不受影响 |
| 预计耗时 | 约 1~2 分钟（7 个源文件、约 20 次模型调用） |

> 只有「短期记忆完整版」和「长期记忆完整版」两节额外依赖 PostgreSQL，其余
> （原版 + 流式 + v3）只需模型。原版用 `MemorySaver` / `InMemoryStore`（纯内存），
> 教学上先看「原理」，再换到 PostgreSQL 看「落库」。

## 本节地图

先看这一课在整章里的位置：`01_模型` 讲了怎么拿到一个模型，`02_消息` 讲了消息结构，
到这一课把「记忆」挂上去、把「流式」打开，后面的「工具 / 智能体 / 中间件」都建立在这之上。

```mermaid
graph TD
    A["智能体的记忆"] --> B["短期记忆<br/>checkpointer"]
    A --> C["长期记忆<br/>store"]
    B --> B1["按 thread_id 隔离<br/>框架自动读写"]
    C --> C1["按 namespace 隔离<br/>自己手动读写"]
    D["流式输出 stream"] --> D1["stream_mode"]
    D1 --> D2["messages 逐 token 打字机"]
    D1 --> D3["updates 看每一步"]
    D1 --> D4["values 看状态快照"]
    D1 --> D5["v3 stream_events<br/>类型化投影"]
```

等价于这张表（裸 JupyterLab 不渲染 mermaid，看表即可）：

| 主题 | 原版（内存） | 完整版（落库） | 核心机制 |
|---|---|---|---|
| 短期记忆 | `MemorySaver` | `PostgresSaver` | `thread_id` 决定是不是同一段会话 |
| 长期记忆 | `InMemoryStore` | `PostgresStore` | `namespace` 决定是谁的记忆，跨会话仍在 |
| 流式 | `messages` / `updates` | `updates` / `messages` / `values` | `stream_mode` 切换观察粒度 |
| 事件流 | — | v3 投影 | `stream_events(version="v3")` 每个投影独立消费 |

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 共享依赖与模型初始化

7 个源文件各自 `import` 了一遍、各自 `init_chat_model` 了一次。合并后这里**统一收口**：
所有 import 放一格，共享一个大模型 `llm`（`streaming=True` 让流式接口生效，对 `invoke` 无副作用）。
事件流 v3 一节因为源文件里变量名是 `model`，会在那一节单独再初始化一次。

In [ ]:
import warnings

from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware
from langchain.chat_models import init_chat_model
from langchain.tools import ToolRuntime
from langchain.tools import tool
from langchain_core.tools import ToolException
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.postgres import PostgresStore
from config import settings

# 共享大模型：短期记忆 / 长期记忆 / 流式输出三节共用同一个实例。
# streaming=True 显式打开流式接口（07 流式一节需要）；对 invoke 无副作用。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    streaming=True,
)

## 前置条件自检

本课有两样外部前提：模型（`api_key`）和 PostgreSQL（`pg_uri`，仅完整版需要）。
这里先探一遍，缺什么提前说清楚；真正连库的验证在「完整版」两节里做（带 try/except）。

In [ ]:
# ===== 前置条件自检：模型与 PostgreSQL 是否就绪 =====
import importlib.util

api_ready = bool(settings.api_key)
pg_cfg_ready = bool(settings.pg_uri)
psycopg_ready = importlib.util.find_spec("psycopg") is not None

print("模型 API_KEY：", "已配置" if api_ready else "未配置（请在 .env 填 API_KEY，本课会真实调用模型）")
print("PostgreSQL PG_URI：", "已配置" if pg_cfg_ready else "未配置（短期/长期记忆完整版会打印[跳过]并降级）")
print("psycopg 驱动：", "已安装" if psycopg_ready else "未安装（Postgres 版记忆会降级）")

## 1. 短期记忆：原版（MemorySaver + thread_id 隔离）

课案原版只有 43 行，讲的是**最短**的短期记忆：`create_agent` 直接配一个
`MemorySaver()`（纯内存 checkpointer），相同 `thread_id` 的对话自动带上历史。

关键点一句话：**每次 `invoke` 只传「本轮新消息」，历史由框架按 `thread_id` 自动取回**
—— 这正是它和 `02_消息` 里「自己 append 到列表」的本质区别。

In [ ]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="你是一个记账助手，帮用户记录并回答账目问题。",
    # LangChain 1.x 支持在 create_agent 里直接配 checkpointer
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "bill-001"}}

r1 = agent.invoke({"messages": [("user", "今天午饭花了 30 元")]}, config)
print("AI：", r1["messages"][-1].content)

r2 = agent.invoke({"messages": [("user", "我今天一共花了多少钱？")]}, config)
print("AI：", r2["messages"][-1].content)

### 预期输出

（待运行后回填真实输出）

## 2. 短期记忆：完整版（PostgresSaver + 落库）

原版用 `MemorySaver`，进程一重启记忆就没了。完整版换成 `PostgresSaver`，把会话状态
**写进 PostgreSQL**，进程重启后只要 `thread_id` 相同还能续聊。

课案原文把连接串写死在代码里，本项目按「铁律 3」换成 `settings.pg_uri`（连接串含口令，绝不进代码）。

短期记忆的三个关键点：

1. **记忆存在 checkpointer 里，不占我们的代码**：每次 `invoke` 只传本轮新消息，历史由框架按
   `thread_id` 自动取回拼在开头；
2. **`thread_id` 是会话号**：同一个 id 才共享历史，换一个 id 就是全新会话 ——
   所以「多用户隔离」只需要给每个用户一个 `thread_id`；
3. **checkpointer 决定「记在哪、活多久」**：

| checkpointer | 存储位置 | 进程重启后 | 适用场景 |
|---|---|---|---|
| `InMemorySaver` | 内存字典 | 丢 | 本地调试、单元测试 |
| `PostgresSaver` | PostgreSQL | 还在（可续聊） | 生产环境（本节演示） |
| `SqliteSaver` | 本地文件 | 还在（单机） | 单机小工具 |

> 前置准备：PostgreSQL 已运行、库名与 `settings.pg_uri` 一致；`checkpointer.setup()` 会自动建表，
> 幂等（表已存在不会报错、也不会清数据）。`PG_URI` 为空或连不上时，`main()` 打印中文提示直接返回，
> **不抛 traceback**。

In [ ]:
# 课案在这里写死了 DB 连接串；本项目从 .env 经 config.py 读取（连接串内含口令，绝不能进代码）
DB_URI = settings.pg_uri


def main() -> None:
    # ---------- 0. 前置检查：连接串没配就直接给中文提示，不抛异常 ----------
    if not DB_URI:
        print("[跳过] settings.pg_uri 为空：请在项目根目录 .env 里配置 PG_URI=")
        print("       postgresql://<用户>:<口令>@127.0.0.1:5432/langgraph")
        return

    try:
        # from_conn_string 是上下文管理器：退出时自动归还连接
        with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
            # ---------- 1. 课案原样：首次运行建表 ----------
            # 幂等操作，表已存在时不会报错，也不会清空已有数据。
            checkpointer.setup()

            # ---------- 2. 把 checkpointer 交给 create_agent ----------
            # 注意 checkpointer 是挂在 agent（底层图）上的，不是挂在某次 invoke 上。
            agent = create_agent(model=llm, tools=[], checkpointer=checkpointer)

            # thread_id="1" 是课案原文用的会话号；
            # config 的位置参数是 invoke 的第二个参数，也可以写成 config=... 关键字。
            config = {"configurable": {"thread_id": "1"}}

            # ---------- 3. 第一轮：只发「我叫张三」 ----------
            # 我们并没有把历史消息带上，但框架会把该 thread 的历史自动拼进去。
            agent.invoke(
                {"messages": [{"role": "user", "content": "我叫张三"}]},
                config,
            )

            # ---------- 4. 第二轮：只发「我叫什么？」 ----------
            # 这一轮之所以能答出来，全靠 checkpointer 按 thread_id 取回了第一轮的消息。
            result = agent.invoke(
                {"messages": [{"role": "user", "content": "我叫什么？"}]},
                config,
            )
            print("===== 4. 课案原文：同一 thread 的跨轮记忆 =====")
            print("AI：", result["messages"][-1].content)   # 预期：你叫张三

            # ---------- 5. 证据：状态里到底存了什么 ----------
            # get_state 读的就是 checkpointer 里该 thread 的最新快照。
            snapshot = agent.get_state(config)
            print("\n===== 5. checkpointer 里存的状态 =====")
            print(f"  该 thread 共 {len(snapshot.values['messages'])} 条消息（不是 1 条！）：")
            for index, msg in enumerate(snapshot.values["messages"], start=1):
                print(f"    [{index}] {msg.type:<7} {str(msg.content)[:40]}")

            # ---------- 6. 换一个 thread_id = 换一个会话（记忆互不干扰） ----------
            # 这是「短期记忆」的边界：只认 thread_id，不认人。
            other_config = {"configurable": {"thread_id": "1-demo-isolated"}}
            result = agent.invoke(
                {"messages": [{"role": "user", "content": "我叫什么？"}]},
                other_config,
            )
            print("\n===== 6. 换 thread_id 后（应当答不出名字） =====")
            print("AI：", result["messages"][-1].content)

            # ---------- 7. 落库证明 ----------
            print("\n===== 7. 存储位置 =====")
            print("  本轮会话已写入 PostgreSQL，连接串来自 settings.pg_uri（不回显口令）")

    except Exception as exc:  # 连接不上数据库时给出可操作的中文提示，而不是一堆 traceback
        print("[跳过] 连接 PostgreSQL 失败：", type(exc).__name__)
        print("       请确认本机 PostgreSQL 已启动、库 langgraph 已创建、.env 的 PG_URI 正确。")
        print("       详情：", str(exc)[:200])


main()

### 预期输出

（待运行后回填真实输出）

## 3. 长期记忆：原版（InMemoryStore + 工具读写）

**长期记忆 = 跨会话共享的记忆（不依赖 `thread_id`）**。在 `create_agent` 里通过 `store`
参数注入，配合工具读写。课案示例场景：智能体记住用户的偏好，新会话里直接使用。

存储结构是**三元组 `(namespace, key, value)`**，例如 `("memories", "1001") → {"name": "小红", "style": "简洁"}`。
本版用 `InMemoryStore`（生产换成 `PostgresStore`），并演示「用两个工具读写长期记忆」。

In [ ]:
store = InMemoryStore()  # 生产环境换成 langgraph.store.postgres.PostgresStore


# ---------- 用工具的方式读写长期记忆 ----------
@tool
def save_user_memory(key: str, value: str) -> str:
    """保存用户的长期记忆。key：记忆条目名（如 name）；value：内容"""
    # 智能体调用工具时通过 store 上下文拿到注入的 store
    store.put(("memories", "1001"), key, {"value": value})
    return f"已保存记忆 {key}={value}"


@tool
def read_user_memory(key: str) -> str:
    """读取用户的长期记忆。key：记忆条目名"""
    item = store.get(("memories", "1001"), key)
    return item.value["value"] if item else "没有这条记忆"


# ---------- 预先写一条记忆，模拟历史会话存下的偏好 ----------
store.put(("memories", "1001"), "style", {"value": "回复风格：简洁，不要废话"})

agent = create_agent(
    model=llm,
    tools=[save_user_memory, read_user_memory],
    system_prompt="你是私人助手。回答前先用 read_user_memory 查用户偏好并遵守。",
    store=store,
)

# 新会话（新 thread_id），但偏好依然生效——这就是「长期」的含义
result = agent.invoke(
    {"messages": [("user", "介绍一下 LangChain")]},
    config={"configurable": {"thread_id": "session-A", "user_id": "1001"}},
)
print("AI：", result["messages"][-1].content)

### 预期输出

（待运行后回填真实输出）

## 4. 长期记忆：完整版（PostgresStore + ToolRuntime）

原版用 `InMemoryStore` 且工具直接抓**闭包里的 `store`**；完整版换成 `PostgresStore`（落库），
工具通过 **`ToolRuntime`** 这个「注入参数」访问 `runtime.store`。

先看「短期记忆 vs 长期记忆」这张对照表，它是本节的分界依据：

| 维度 | 短期记忆（checkpointer） | 长期记忆（store） |
|---|---|---|
| 存放内容 | 整个会话的消息历史 | 你主动写进去的事实 / 偏好 / 知识 |
| 作用域 | `thread_id`（会话级） | `namespace`（按用户、按业务自定义） |
| 谁写入 | 框架自动写 | **必须自己写**（工具或中间件里 `put`） |
| 谁读出来 | 框架自动拼进 messages | **必须自己读**（工具里 `search` / `get`） |
| 跨会话 | 换 `thread_id` 就没了 | 换 `thread_id` 依然在 |

**为什么长期记忆要「手动」？** 因为「什么值得长期记住」是业务判断，框架替你决定不了：
用户说「我讨厌香菜」是该记的，说「今天几号」就不该记。所以课案把它做成两个工具，
让**模型自己决定**何时记住、何时回忆。

存储结构三元组 `(namespace, key, value)`：

- `namespace = ("memories", "u1")` —— 命名空间，元组形式，第一段通常是类别，第二段是用户/租户；
- `key = "info"` —— 条目名，同一 namespace 下唯一，重复 `put` 是覆盖；
- `value = {"data": "..."}` —— 任意 JSON（dict），这里约定用一个 `data` 字段装正文。

In [ ]:
# ---------- 1. 课案原文的两个工具：写记忆 / 读记忆 ----------
# 关键点：ToolRuntime 是「注入参数」，不是模型要填的参数 ——
# 所以工具 schema 里只会出现 info / query，模型看不到 runtime 这一项。
# runtime 上挂着 store、state、config 等运行时上下文，是工具访问长期记忆的唯一入口。
@tool
def remember(info: str, runtime: ToolRuntime) -> str:
    """记住用户信息"""
    # namespace 固定为 ("memories", "u1")：课案用一个假用户 id 演示
    runtime.store.put(("memories", "u1"), "info", {"data": info})
    return f"已记住: {info}"


@tool
def recall(query: str, runtime: ToolRuntime) -> str:
    """回忆用户之前的信息"""
    # search = 按自然语言查询；配了 embedding 的 store 会做语义检索，
    # 没配 embedding 时退化为「列出该 namespace 下的条目」（本项目即如此）。
    # 所以下面这句「or "暂无"」不是多余的：search 返回空列表时 join 出来是空串，
    # 模型会收到一个空白 ToolMessage，容易误判成「记忆功能坏了」。
    mems = runtime.store.search(("memories", "u1"), query=query, limit=3)
    return "；".join([m.value["data"] for m in mems]) or "暂无"


def main() -> None:
    if not settings.pg_uri:
        print("[跳过] settings.pg_uri 为空：长期记忆的 PostgresStore 需要 PostgreSQL。")
        print("       请在项目根目录 .env 里配置 PG_URI=postgresql://<用户>:<口令>@127.0.0.1:5432/langgraph")
        return

    try:
        # ---------- 2. 课案原文：把 store 挂到 agent 上 ----------
        with PostgresStore.from_conn_string(settings.pg_uri) as store:
            store.setup()   # 首次运行建表（幂等）

            # 模拟「上一个会话早就存下来的记忆」：
            # 注意这一步发生在 agent 运行之前，说明记忆是先于对话存在的。
            store.put(("memories", "u1"), "info", {"data": "用户叫张三，25岁"})

            # 课案原文是 create_agent(model=model, tools=[remember, recall], store=store)。
            # 这里只多补了一句 system_prompt：本机模型比较"随性"，
            # 不明确要求它「先查记忆再回答」时经常直接凭人设作答，演不出长期记忆的效果。
            agent = create_agent(
                model=llm,
                tools=[remember, recall],
                store=store,
                system_prompt=(
                    "你是私人助理。回答任何关于用户个人信息（姓名、年龄、住址、喜好）的问题前，"
                    "必须先调用 recall 工具查询长期记忆，不要凭空回答；"
                    "用户要求你记住某事时调用 remember 工具。"
                ),
            )

            # ---------- 3. 课案原文调用：让模型自己决定用 recall 工具 ----------
            result = agent.invoke(
                {"messages": [{"role": "user", "content": "我叫什么？多大？"}]}
            )
            print("===== 3. 模型凭长期记忆回答（没有历史消息） =====")
            print("AI：", result["messages"][-1].content)
            print("\n  完整轨迹（期望能看到模型主动调用了 recall）：")
            for index, msg in enumerate(result["messages"], start=1):
                print(f"    [{index}] {msg.type:<7} {str(msg.content)[:60]}")

            called_recall = any(
                getattr(m, "tool_calls", None) and any(c["name"] == "recall" for c in m.tool_calls)
                for m in result["messages"]
            )
            if not called_recall:
                # 本机模型偶发「不调工具直接作答」，这不是代码问题，提示一下避免误判
                print("  ⚠ 本轮模型没有调用 recall（本机模型偶发行为），上面的回答不代表长期记忆失效。")
                print("    下方第 4 步直接读写 store，可以无条件验证记忆确实存着。")

            # ---------- 4. 直接验证 store 的增删改查（不经过模型） ----------
            # 第 3 步依赖模型「愿意调工具」，这一步绕开模型直接读写 store ——
            # 教学上很重要：把「存储是否正常」和「模型是否听话」两件事拆开验证，
            # 否则模型不发 tool_calls 时你会误以为是长期记忆坏了。
            print("\n===== 4. 直接操作 store =====")
            store.put(("memories", "u1"), "hobby", {"data": "用户喜欢打羽毛球"})
            item = store.get(("memories", "u1"), "hobby")
            print("  get(('memories','u1'), 'hobby') →", item.value if item else None)

            mems = store.search(("memories", "u1"), query="用户的爱好", limit=3)
            print("  search(query='用户的爱好') →", [(m.key, m.value["data"]) for m in mems])

            # ---------- 5. 换 namespace = 换一个人的记忆 ----------
            # 长期记忆按 namespace 隔离，和 thread_id 无关：
            # 也就是「换会话仍在」，这正是「长期」二字的含义。
            store.put(("memories", "u2"), "info", {"data": "用户叫李四，30岁"})
            print("\n===== 5. namespace 隔离 =====")
            print("  u1 →", [m.value["data"] for m in store.search(("memories", "u1"), limit=10)])
            print("  u2 →", [m.value["data"] for m in store.search(("memories", "u2"), limit=10)])

            # ---------- 6. 让模型真的「写入」一条新记忆 ----------
            # 注意这次 invoke 没有传 checkpointer，所以模型手里**没有**第 3 步的对话历史；
            # 它仍然能被记住，全靠 store 跨会话存在 —— 这就是长期记忆与短期记忆的分界。
            print("\n===== 6. 让模型调用 remember 工具写新记忆 =====")
            result = agent.invoke(
                {"messages": [{"role": "user", "content": "记住：我住在南昌，平时用 Python 写后端。"}]}
            )
            print("AI：", result["messages"][-1].content)
            print("  写入后 u1 的全部记忆：",
                  [m.value["data"] for m in store.search(("memories", "u1"), limit=10)])

    except Exception as exc:
        print("[跳过] 连接 PostgreSQL 失败：", type(exc).__name__)
        print("       请确认本机 PostgreSQL 已启动、库 langgraph 已创建、.env 的 PG_URI 正确。")
        print("       详情：", str(exc)[:200])


main()

### 预期输出

（待运行后回填真实输出）

## 5. 流式输出：原版（messages / updates）

原版只有 55 行，演示两种流式：

1. **消息流（token 打字机）**：`stream_mode="messages"`，逐 token 吐文本；
2. **步骤流**：`stream_mode="updates"`，看到每个节点 / 工具的进度。

注意模型要带 `streaming=True` 初始化（共享 `llm` 已经设了）。

In [ ]:
@tool
def get_time() -> str:
    """获取当前时间"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


agent = create_agent(model=llm, tools=[get_time])

print("===== token 流式 =====")
for token, metadata in agent.stream(
    {"messages": [("user", "现在几点了？顺便问一下你是什么模型")]},
    stream_mode="messages",
):
    # 只打印 AI 的 token（过滤掉工具消息）
    if token.content and metadata.get("langgraph_node") == "model":
        print(token.content, end="", flush=True)
print()

print("===== 步骤流式 =====")
for update in agent.stream(
    {"messages": [("user", "现在几点了？")]},
    stream_mode="updates",
):
    print(update)

### 预期输出

（待运行后回填真实输出）

## 6. 流式输出：完整版（updates / messages / values）

完整版把 `stream_mode` 的几种取值讲全，并换了一个「天气」工具演示。

**为什么需要流式？** 智能体一次 `invoke` 可能要跑好几秒（模型思考 → 调工具 → 再思考），
全部跑完再一次性返回，用户会觉得「卡住了」。流式把中间过程一点点吐出来。

`stream_mode` 的几种取值（同一份 agent，换个参数就能切换观察粒度）：

| stream_mode | 每次吐出什么 | 典型用途 |
|---|---|---|
| `"values"` | 状态的**完整快照** | 想看全局状态怎么长出来的 |
| `"updates"` | 本次只被**改动**的那部分（默认值） | 看节点 / 工具的执行顺序（课案用它） |
| `"messages"` | `(消息分片, 元数据)` 二元组，**逐 token** | 打字机效果、前端流式渲染 |
| `"custom"` | 你自己在节点里 emit 的自定义数据 | 自定义进度条 |
| `"debug"` | 最啰嗦的调试事件 | 排查框架内部行为 |

两条容易踩的坑：

1. `stream_mode="updates"` 吐的是 `{节点名: 该节点的输出}`，所以要写**两层循环**：
   外层是每一步，内层 `.items()` 拆出节点名和数据；
2. `stream_mode="messages"` 吐的是**分片**而不是完整消息，工具调用的参数也会以分片形式混进来；
   所以要先按 `metadata["langgraph_node"] == "model"` 过滤，再判断 `content` 非空。

In [ ]:
# ---------- 1. 课案原文的天气工具 ----------
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气。"""
    return f"{city}永远是晴天！"


agent = create_agent(model=llm, tools=[get_weather])

# 同一份输入复用三次：三个 stream_mode 观察的是**同一次执行的不同侧面**，
# 传同一个 question 才好横向对比（注意是三次独立运行，不是同一次跑三遍）。
question = {"messages": [{"role": "user", "content": "北京天气怎么样？"}]}

# ---------- 2. 课案原文：stream_mode="updates"（按步骤流） ----------
print("===== 2. 课案原文：stream_mode='updates' —— 看每一步 =====")
for chunk in agent.stream(question, stream_mode="updates"):
    # chunk 形如 {"model": {"messages": [AIMessage(...)]}} 或 {"tools": {...}}
    for step, data in chunk.items():
        msgs = data.get("messages", [])
        if msgs:
            last = msgs[-1]
            # content_blocks 是 LangChain 1.x 的结构化内容块（文本 + 工具调用一起）；
            # 老版本 / 纯文本消息没有这个属性，所以用 hasattr 兜底。
            text = last.content_blocks if hasattr(last, "content_blocks") else str(last.content)
            print(f"[{step}] {text}")

# ---------- 3. stream_mode="messages"：逐 token 打字机 ----------
print("\n===== 3. stream_mode='messages' —— 打字机效果 =====")
for token, metadata in agent.stream(question, stream_mode="messages"):
    # 过滤：只要模型节点吐出来的正文；工具节点的分片和空分片都跳过。
    if token.content and metadata.get("langgraph_node") == "model":
        print(token.content, end="", flush=True)   # flush 保证立刻显示，不被行缓冲吞掉
print()   # 收尾换行

# ---------- 4. stream_mode="values"：看状态快照的演变 ----------
print("\n===== 4. stream_mode='values' —— 状态快照 =====")
for snapshot in agent.stream(question, stream_mode="values"):
    # 每个快照都是「此刻的完整状态」，所以消息条数会单调增长：1 → 2 → 3 → 4
    print(f"  快照消息数：{len(snapshot['messages'])}，"
          f"最新一条：{type(snapshot['messages'][-1]).__name__}")

# ---------- 5. 异步流式（Web 服务里常用） ----------
# FastAPI / WebSocket 场景下用 astream，边收边推给前端：
#     async for token, metadata in agent.astream(question, stream_mode="messages"):
#         await websocket.send_text(token.content)
# 本节故意**只演示同步版**：逻辑与异步版一模一样，只是 for / async for 之差，
# 真跑异步还得额外起 asyncio 事件循环，反而盖住了「流式」这个主题。
print("\n===== 5. 异步流式 =====")
print("  agent 同时提供 ainvoke / astream，接口与同步版一一对应（见文件头注释示例）")

### 预期输出

（待运行后回填真实输出）

## 7. 官方补充：事件流 v3（stream_events）

这一节对照 LangChain **官方文档** /oss/python/langchain/event-streaming，补上 `07_流式`
之后的另一半：**官方现在推荐新项目用 v3 事件流**。

官方原文（第一段就给了结论）：

> For most application and frontend use cases, use **Event Streaming** through
> `stream_events(..., version="v3")`. Event Streaming returns a run object with
> typed projections, so each projection can be consumed independently instead of
> parsing stream-mode tuples.

课案 `07_流式` 讲的是 `stream_mode`（values / updates / messages / custom 等），那套仍然可用；
v3 的差别是**把每种数据做成独立投影**，不必再去解析元组：

| 投影 | 用途 |
|---|---|
| `for event in stream` | 最原始的协议事件（全信封、所有通道） |
| `stream.messages` | 模型消息流，每次 LLM 调用一个 |
| `message.text` | 文本增量（逐 token） |
| `message.reasoning` | 推理内容增量（模型支持时才有） |
| `message.tool_calls` | 工具调用入参的增量与最终结果 |
| `message.output` | 模型调用完成后的完整消息对象 |
| `stream.values` | agent 状态快照 |
| `stream.output` | 最终状态 |
| `stream.subgraphs` | 嵌套子图运行 |
| `stream.subagents` | **命名**子代理的运行（可拿到内层消息） |
| `stream.extensions` | 自定义 transformer 投影 |
| `stream.tool_calls` | 工具执行生命周期（入参/输出增量/输出/错误） |

⚠️ **两条本机实测要点**（官方文档没放在显眼处）：

- **A. v3 目前是实验性协议**：本地 langchain 1.4.0 / langgraph 1.2.11 调用时会打印
  `LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental` ——
  能在生产用，但要预期 API 变动；
- **B. 投影是单次消费的**：同一个 stream 上先把 `stream.messages` 抽干，
  再读 `stream.tool_calls` 会拿到空结果。多个投影要么各开一个新 stream，
  要么用 `stream.interleave(...)` 一次交织消费（Demo 2 实测对比了这两种写法）。

对应 `Agent/官方文档缺口对照.md` 的 **LangChain 第 6 项**（事件流 v3）。

In [ ]:
# v3 是实验性协议，每次调用都会打 Beta 警告；这里统一静音以免刷屏，
# 但**请知道它的存在**（升级 langgraph 时优先回归测试本文件）。
warnings.filterwarnings("ignore", category=Warning, module="langgraph")

model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

INPUT = {"messages": [{"role": "user", "content": "北京天气如何？用一句话回答。"}]}


@tool
def get_weather(city: str) -> str:
    """查询城市天气。"""
    return f"{city}：晴，25℃"


@tool
def risky_lookup(keyword: str) -> str:
    """查询一个可能不存在的东西（演示工具报错在流里怎么体现）。"""
    # 注意（实测结论）：**普通异常与 ToolException 一样，都会让运行直接中断** ——
    # ToolNode 的默认处理器只把 `ToolInvocationError` 转成错误消息，其余一律 re-raise
    # （源码 langgraph/prebuilt/tool_node.py 的 _default_handle_tool_errors）。
    # 本 Demo 正是要演示这一点：没挂 ToolErrorMiddleware 时，失败根本进不了流。
    raise ToolException(f"下游服务查不到 {keyword!r}")


def build_weather_agent():
    return create_agent(model=model, tools=[get_weather])

### 7.1 Demo 1：基础 —— 逐 token 文本、finalize 的消息对象、token 用量

`stream.messages` 每次 LLM 调用吐一个 `message`；`message.text` 是增量流（打字机），
`message.output` 是 finalize 后的完整 `AIMessage`（能拿到 `usage_metadata` 里的 token 用量）。

In [ ]:
def demo_1_basic_projections() -> None:
    print("=" * 70)
    print("Demo 1：stream.messages —— 逐 token 文本 + 完整消息 + token 用量")
    print("=" * 70)

    agent = build_weather_agent()
    stream = agent.stream_events(INPUT, version="v3")

    for index, message in enumerate(stream.messages, start=1):
        print(f"\n  第 {index} 次模型调用（节点：{message.node}）")
        # .text 是可迭代的增量流；边收边打印就是打字机效果
        pieces = []
        for delta in message.text:
            pieces.append(delta)
        text = "".join(pieces)
        print(f"    文本增量拼接：{text[:90]!r}" if text else "    （本次没有文本输出，只产出了工具调用）")

        # .output 是 finalize 后的完整 AIMessage
        final = message.output
        usage = getattr(final, "usage_metadata", None)
        if usage:
            print(f"    token 用量：输入 {usage.get('input_tokens')} / 输出 {usage.get('output_tokens')}"
                  f" / 合计 {usage.get('total_tokens')}")
            details = usage.get("output_token_details") or {}
            if details.get("reasoning"):
                print(f"    其中推理 token：{details['reasoning']}（该模型会产出 reasoning）")

        # reasoning 投影：模型不吐推理内容时这里是空的
        reasoning = "".join(delta for delta in message.reasoning)
        print(f"    reasoning 投影长度：{len(reasoning)} 字符")

        # 工具调用投影（本条消息发起的调用）
        finalized = message.tool_calls.get() if hasattr(message.tool_calls, "get") else None
        if finalized:
            print(f"    本条消息发起的工具调用：{[(c['name'], c['args']) for c in finalized]}")

    print(f"\n  最终状态里的最后一条消息：{str(stream.output['messages'][-1].content)[:90]}")
    print(
        "  ↑ 与课案 07 的 stream_mode='messages' 相比，v3 把「文本 / 推理 / 工具调用 / 完整消息 /\n"
        "    token 用量」拆成了同一对象上的不同属性，前端按需取用，不必再解析元组。"
    )


demo_1_basic_projections()

### 预期输出

（待运行后回填真实输出）

### 7.2 Demo 2：投影的消费规则（本文件最值钱的一课）

实测现象：

- ① 先 `for m in stream.messages`（抽干）→ 再 `list(stream.tool_calls)` → **空**；
- ② 单开一个新 stream 只读 `tool_calls` → 正常拿到；
- ③ `stream.interleave("messages", "tool_calls", "values")` → 一次拿到所有投影的事件。

结论：投影是**同一条底层事件流的多个视图**，谁先被抽干谁就把事件消费掉了。

In [ ]:
def demo_2_consumption_rules() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：投影的单次消费规则（错误写法 vs 正确写法）")
    print("=" * 70)

    agent = build_weather_agent()

    # ---- 错误写法：同一个 stream 上依次消费两个投影 ----
    stream = agent.stream_events(INPUT, version="v3")
    message_count = sum(1 for _ in stream.messages)
    leftover_calls = len(list(stream.tool_calls))
    print(f"  错误写法（先 messages 后 tool_calls）：messages={message_count} 条，"
          f"tool_calls={leftover_calls} 个 ← 被抽干了")

    # ---- 正确写法 A：每个投影开一个新 stream（最简单）----
    stream_a = agent.stream_events(INPUT, version="v3")
    calls_a = list(stream_a.tool_calls)
    print(f"  正确写法 A（各开新 stream）：tool_calls={len(calls_a)} 个")

    # ---- 正确写法 B：interleave 一次交织消费（推荐，只跑一次图）----
    stream_b = agent.stream_events(INPUT, version="v3")
    counts: dict[str, int] = {}
    for kind, _payload in stream_b.interleave("messages", "tool_calls", "values"):
        counts[kind] = counts.get(kind, 0) + 1
    print(f"  正确写法 B（interleave）：{counts}")
    print(
        "  ↑ 记这一条就够了：**想同时要多种投影，就用 interleave**；\n"
        "    各开新 stream 虽然简单，但每一路都会把图重跑一遍（模型要重复付费）。"
    )


demo_2_consumption_rules()

### 预期输出

（待运行后回填真实输出）

### 7.3 Demo 3：工具执行生命周期 —— 入参、输出、以及 error 字段

`stream.tool_calls` 能拿到 `tool_name` / `input` / `output` / `error`。三个实测细节：

1. 正常调用的 `output` 是 **ToolMessage**，但必须把流消费完再读（循环里边收边读是 `None`）；
2. 工具抛 `ToolException` 且**没挂**错误中间件时，运行直接中断、流里看不到失败事件；
3. 挂上 `ToolErrorMiddleware` 后，同一失败变成 `output=None, error='下游服务查不到 ...'` ——
   失败成为可观测的一等事件。

In [ ]:
def demo_3_tool_lifecycle() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：stream.tool_calls —— 工具执行生命周期（含失败）")
    print("=" * 70)

    agent = create_agent(model=model, tools=[get_weather, risky_lookup])

    # ---- A. 正常调用：注意要把流**消费完**再读 output ----
    stream = agent.stream_events(INPUT, version="v3")
    calls = list(stream.tool_calls)      # 先抽干，事件才会 finalize
    for call in calls:
        print(f"  ✔ 正常调用：{call.tool_name}({call.input})")
        print(f"     输出类型：{type(call.output).__name__}，内容：{str(call.output)[:60]}")
        print(f"     error 字段：{call.error}")
    print(
        "     实测提醒：在 for 循环里边收边读 `call.output` 会拿到 **None** ——\n"
        "     投影是按生命周期推进的，必须把流消费完（或先 list() 收集）后再读最终值。"
    )

    # ---- B. 工具失败但没有错误处理中间件 ----
    print("\n  --- B. 工具抛 ToolException，但**没有**错误处理中间件 ---")
    stream = agent.stream_events(
        {"messages": [{"role": "user", "content": "查一下 keyword='不存在的记录'"}]},
        version="v3",
    )
    try:
        list(stream.tool_calls)
        print("     竟然没抛异常？（不符合预期）")
    except Exception as exc:  # noqa: BLE001
        print(f"     运行直接中断：{type(exc).__name__}: {str(exc)[:70]}")
    print(
        "     实测：本版本里 `ToolException` 也会被 ToolNode 的默认处理**原样抛出**，\n"
        "     整个运行中断 —— 流里自然也就看不到任何「工具失败」事件。"
    )

    # ---- C. 同一个失败工具，挂上 ToolErrorMiddleware ----
    print("\n  --- C. 同一个失败工具，挂 ToolErrorMiddleware ---")

    def on_error(exc: Exception, request) -> str:
        """把工具异常转成模型能读懂的说明（返回 None 则让异常继续往外抛）。

        注意签名是**两个参数**：(exc, request) —— request.tool_call 里有工具名/入参/id，
        需要按工具分流处理时用它（写成单参数会报 TypeError，本文件实测踩过）。
        """
        print(f"     [on_error] 捕获 {type(exc).__name__}：{str(exc)[:50]}")
        return f"工具执行失败（{type(exc).__name__}）。请换个关键词或直接说明无法查询。"

    safe_agent = create_agent(
        model=model,
        tools=[get_weather, risky_lookup],
        middleware=[ToolErrorMiddleware(on_error)],
    )
    stream = safe_agent.stream_events(
        {"messages": [{"role": "user", "content": "查一下 keyword='不存在的记录'"}]},
        version="v3",
    )
    for call in list(stream.tool_calls):
        print(f"     失败调用：{call.tool_name}({call.input})")
        print(f"       输出：{str(call.output)[:70]}")
        print(f"       error 字段：{call.error}")
    print(
        "  ↑ 对照很清楚：**不挂中间件 → 运行中断；挂上 → 失败变成一条普通的工具事件**，\n"
        "    流能正常跑完、模型也能读到失败原因并换个办法。\n"
        "    这也是官方错误处理四分类里「LLM 可修复」那一类的落地方式\n"
        "    （中间件细节见 02_langchain/11_内置中间件_官方补充.py Demo 1）。"
    )


demo_3_tool_lifecycle()

### 预期输出

（待运行后回填真实输出）

### 7.4 Demo 4：子代理投影 —— 命名子代理的运行能单独看

官方说明：内层 agent 通过**包裹工具**被调用时，它的事件落在嵌套命名空间；
你在 `create_agent(name=...)` 里给的名字就是流里的标识，而 `.cause` 是派发它的那次工具调用。

In [ ]:
def demo_4_subagents() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：stream.subagents —— 命名子代理单独成流")
    print("=" * 70)

    inner = create_agent(
        model=model,
        tools=[get_weather],
        system_prompt="你只回答天气问题，一句话以内。",
        name="weather_expert",          # ← 这个名字决定它在流里的标识
    )

    @tool
    def ask_weather_expert(question: str) -> str:
        """把天气问题转给天气专家代理。"""
        result = inner.invoke({"messages": [{"role": "user", "content": question}]})
        return str(result["messages"][-1].content)

    outer = create_agent(model=model, tools=[ask_weather_expert])
    stream = outer.stream_events(
        {"messages": [{"role": "user", "content": "帮我问天气专家：上海天气怎么样？"}]},
        version="v3",
    )

    # 内联消费：边拿子代理句柄边读它自己的投影（不要在循环外先 list()，理由见 Demo 2）
    inner_texts: list[str] = []
    for sub in stream.subagents:
        cause = getattr(sub, "cause", None)
        # 实测：cause 是个 dict（形如 {'type': 'toolCall', 'tool_call_id': ...}），
        # 里面**没有**工具名字段，要拿工具名得回到消息里按 tool_call_id 反查
        cause_id = cause.get("tool_call_id") if isinstance(cause, dict) else None
        print(f"  子代理 name={sub.name!r}，由工具调用 id={cause_id} 派发")
        for message in sub.messages:
            text = "".join(delta for delta in message.text) or str(message.output.content)
            inner_texts.append(text[:80])
            print(f"    内层模型输出：{text[:80]!r}")

    if not inner_texts:
        print("  内层 messages 为空 —— 说明句柄拿到了但内层消息没被投影出来（见文末实测结论）")
    print(
        "  ↑ 子代理投影的价值：多 Agent 系统里，主代理与子代理的输出能**分别**喂给前端 ——\n"
        "    用户既能看到「主代理在等子代理」，也能看到子代理自己说了什么。"
    )


demo_4_subagents()
print("\n全部 Demo 执行完毕。")

### 预期输出

（待运行后回填真实输出）

### 7.5 v3 实测结论 / 未收录 / 踩坑（原文存档）

**1. 实测结论（langchain 1.4.0 / langgraph 1.2.11，本机）：**

- `agent.stream_events(input, version="v3")` 返回 `GraphRunStream`，可用投影：
  `abort / extensions / interleave / interrupted / interrupts / lifecycle / messages / output /
  subagents / subgraphs / tool_calls / values`；
- 调用会打印 `LangChainBetaWarning`（v3 是实验性协议），本文件已统一静音；
- Demo 1：`message.text` 能拼出完整文本；`message.output.usage_metadata` 给出
  输入 / 输出 / 合计 token 与推理 token（本机模型实测 `reasoning=208`）；
- Demo 2：同一 stream 上先消费 `messages` 后，`tool_calls` 拿到 **0** 个（被抽干）；
  新开 stream 或 `interleave` 都能正常拿到（`interleave` 实测交织出 7 个事件）；
- Demo 3：`stream.tool_calls` 能拿到 `tool_name / input / output / error`。三个实测细节：
  ① 正常调用的 `output` 是 **ToolMessage**，但必须把流消费完再读（循环里边收边读是 `None`）；
  ② 工具抛 `ToolException` 且**没挂**错误中间件时，运行直接中断、流里看不到失败事件；
  ③ 挂上 `ToolErrorMiddleware` 后同一失败变成 `output=None, error='下游服务查不到 ...'` ——
  失败成为可观测的一等事件；
- Demo 4：`stream.subagents` 能按 `name=` 对齐子代理；**内联消费**（边拿句柄边读
  `sub.messages`）能拿到内层 2 条模型消息（1 次工具调用 + 1 次文本），
  而先 `list()` 收集句柄再读则为空 —— 与 Demo 2 的消费规则同源；
  `cause` 是形如 `{'type': 'toolCall', 'tool_call_id': ...}` 的 dict，**不含工具名**。

**2. 未收录（官方还有、本文件没做的）：**

- `stream.extensions`（自定义 transformer 投影）：要自己实现 transformer 插件，属于前端/协议层扩展；
- `stream.subgraphs`（普通嵌套子图的运行流）：与 `subagents` 投影的差别是「未命名的子图」；
- 异步接口 `astream_events`：本文件只用同步写法，异步用法与 v2 一致；
- v2 `astream_events` / `stream_mode` 的对照迁移：课案 `07_流式` 已覆盖 `stream_mode`。

**3. 踩坑提示：**

- A. **投影单次消费**：想同时要多种投影 → 用 `interleave(...)`；各开新 stream 会让图重跑（模型重复计费）；
- B. v3 的 `message.output.content` 可能是**内容块列表**（text / tool_call 等），不是纯字符串 ——
  本文件取文本时用了 `"".join(message.text)`，别直接对 content 做字符串操作；
- C. v3 是实验性协议，升级 langgraph 后请优先回归本文件；
- D. 工具报错**不管抛哪种异常，默认都会直接中断运行**（ToolNode 只把 `ToolInvocationError`
  转成错误消息，普通异常与 `ToolException` 一律 re-raise）；要让「工具失败」变成流里可见的事件，
  必须挂 `ToolErrorMiddleware`（Demo 3 Part C 就是对照）；
- E. 子代理要出现在 `stream.subagents` 里，必须给内层 `create_agent(name=...)` 命名；
  不命名则是普通子图（落在 `subgraphs` 投影）。读内层消息同样要**内联消费**；
- F. `ToolErrorMiddleware(on_error)` 的 `on_error` 是**两参数** `(exc, request)` ——
  写成单参数会在工具报错时抛 TypeError（本文件实测踩过）。

## 小结

- **短期记忆**靠 `checkpointer`，按 `thread_id` 隔离，历史由框架自动取回、自动拼接；
- **长期记忆**靠 `store`，按 `namespace` 隔离，跨会话仍在，但**读和写都得自己来**（工具里 `put` / `search`）；
- 两者都可选「内存版」（调试用）或「Postgres 版」（生产用），接口几乎一致，只换一个类名；
- **流式**用 `stream_mode` 切换观察粒度：`messages` 逐 token、`updates` 看步骤、`values` 看快照；
- **事件流 v3**（`stream_events(version="v3")`）把每种数据做成独立投影，前端按需取用，
  但**投影是单次消费的**，要多种投影就 `interleave(...)`；
- 工具失败默认会中断运行，要让它变成可观测事件，挂 `ToolErrorMiddleware`。

下一课 `04_中间件_钩子与人工审核.ipynb` 会展开中间件与人工审核 —— 正是本节
`ToolErrorMiddleware` 那类「错误处理中间件」的完整落地。

## 常见坑

1. **`MemorySaver` 进程一重启记忆就没了** —— 本地调试没问题，上线要换 `PostgresSaver`；
2. **换 `thread_id` 就失忆**：这不是 bug，短期记忆的边界就是会话号；
  多用户隔离 = 每个用户一个 `thread_id`；
3. **长期记忆忘了「自己读写」**：`store` 不会自动写，工具里不 `put` / `search`，
  模型就是「没有记忆」；
4. **`recall` 返回空时忘了兜底**：`"；".join([])` 是空串，模型会收到空白 ToolMessage，
  容易误判「记忆坏了」，所以要 `or "暂无"`；
5. **流式 `updates` 要两层循环**：外层 `for chunk`、内层 `chunk.items()`，少一层会打印一堆
  `dict` 而不是节点名；
6. **流式 `messages` 要过滤**：分片里混着工具调用，只认 `metadata["langgraph_node"] == "model"`
  且 `content` 非空；
7. **v3 投影单次消费**：同一 stream 上先后读两个投影，第二个会拿到空结果 —— 用 `interleave`；
8. **工具报错默认中断运行**：不管抛 `ToolException` 还是普通异常，要「失败可见」就挂
  `ToolErrorMiddleware`（`on_error` 是两参数 `(exc, request)`，写单参数会 TypeError）。

## 官方链接

- 短期记忆：<https://docs.langchain.com/oss/python/langchain/short-term-memory>
- 长期记忆：<https://docs.langchain.com/oss/python/langchain/long-term-memory>
- 流式输出：<https://docs.langchain.com/oss/python/langchain/streaming>
- 事件流 v3：<https://docs.langchain.com/oss/python/langchain/event-streaming>
- 智能体（create_agent）：<https://docs.langchain.com/oss/python/langchain/agents>